# Skin Lesion Classification with Transfer Learning
## ResNet50 vs. VGG16

This notebook transcribes and organizes the code from **Appendix C** of the accompanying research paper comparing ResNet50 and VGG16 for skin-lesion classification.

The appendix code has been cleaned only for formatting (line wraps, spacing, and indentation) so it is readable as executable Python. The underlying logic and original paths are preserved.

### Important reproducibility notes
- Appendix C contains a **C.3 heading for accuracy calculation but no separate code block** beneath it.
- Appendix C.4 imports `confusion_matrix`, but the printed appendix does not actually compute or display confusion matrices.
- The appendix split code yields approximately **75% training / 15% validation / 10% test**, even though the prose methodology describes a different split.
- The appendix training code sets a maximum of **25 epochs** with early stopping.

At the end of the notebook, a clearly marked **Supplementary evaluation** cell adds the missing accuracy and confusion-matrix calculations. That cell is not part of the original appendix code.

## C.1 - Data preprocessing and dataset organization

In [ ]:
import os
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split

# Define paths
metadata_path = "C:/Users/vivaa/Downloads/ham10000_metadata_2024-02-11.csv"
images_directory = r"C:\Users\vivaa\Downloads\ISIC-images"
dataset_directory = r"C:\Users\vivaa\Desktop\IB\Computer Science HL\Extended essay\Dataset"

# Read the metadata file
metadata = pd.read_csv(metadata_path)

train_val, test = train_test_split(
    metadata,
    test_size=0.10,
    stratify=metadata['diagnosis'],
    random_state=42,
)

train, val = train_test_split(
    train_val,
    test_size=0.1667,
    stratify=train_val['diagnosis'],
    random_state=42,
)

# Organize the dataset
def organize_dataset(dataset_split, dataset_split_name):
    for _, row in dataset_split.iterrows():
        src_file = os.path.join(images_directory, row['isic_id'] + '.jpg')

        dest_dir = os.path.join(
            dataset_directory,
            dataset_split_name,
            row['diagnosis'],
        )
        if not os.path.exists(dest_dir):
            os.makedirs(dest_dir)

        dest_file = os.path.join(dest_dir, row['isic_id'] + '.jpg')
        shutil.copy(src_file, dest_file)

# Apply the organization function to each split
organize_dataset(train, 'train')
organize_dataset(val, 'val')
organize_dataset(test, 'test')

print("Dataset organization complete.")

### Split implied by the appendix code

The first split reserves 10% of the full dataset for testing. The second split assigns 16.67% of the remaining 90% to validation, which is approximately 15% of the original dataset. The resulting split is therefore approximately **75% train / 15% validation / 10% test**.

## C.2 - Train ResNet50 and VGG16

In [ ]:
import numpy as np
import tensorflow as tf
from keras.preprocessing.image import ImageDataGenerator
from keras.applications import ResNet50, VGG16
from keras.layers import Dense, GlobalAveragePooling2D
from keras.models import Model
from keras.callbacks import EarlyStopping, ModelCheckpoint

IMG_WIDTH, IMG_HEIGHT = 224, 224
BATCH_SIZE = 32

train_dir = '/content/Dataset/train'
val_dir = '/content/Dataset/val'
test_dir = '/content/Dataset/test'

train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
)

val_test_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_WIDTH, IMG_HEIGHT),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
)

val_generator = val_test_datagen.flow_from_directory(
    val_dir,
    target_size=(IMG_WIDTH, IMG_HEIGHT),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,
)

test_generator = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_WIDTH, IMG_HEIGHT),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,
)


def create_model(base_model, num_classes):
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    predictions = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs=base_model.input, outputs=predictions)
    return model


num_classes = len(train_generator.class_indices)

resnet_base = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_WIDTH, IMG_HEIGHT, 3),
)
resnet_model = create_model(resnet_base, num_classes)

vgg_base = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_WIDTH, IMG_HEIGHT, 3),
)
vgg_model = create_model(vgg_base, num_classes)

resnet_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

vgg_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

early_stopping = EarlyStopping(monitor='val_loss', patience=10)
resnet_checkpoint = ModelCheckpoint(
    'resnet_model.keras',
    monitor='val_loss',
    save_best_only=True,
)
vgg_checkpoint = ModelCheckpoint(
    'vgg_model.keras',
    monitor='val_loss',
    save_best_only=True,
)

history_resnet = resnet_model.fit(
    train_generator,
    epochs=25,
    validation_data=val_generator,
    callbacks=[early_stopping, resnet_checkpoint],
)

history_vgg = vgg_model.fit(
    train_generator,
    epochs=25,
    validation_data=val_generator,
    callbacks=[early_stopping, vgg_checkpoint],
)

## C.3 - Accuracy calculation

The appendix includes this section heading but does **not** provide a separate accuracy-calculation code block. The supplementary evaluation section below adds an explicit accuracy calculation without presenting it as original appendix code.

## C.4 - Weighted F1 evaluation from the appendix

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, confusion_matrix
from keras.models import load_model
from keras.preprocessing.image import ImageDataGenerator
import pandas as pd

model1 = load_model('/content/drive/MyDrive/Models/resnet_model.keras')
model2 = load_model('/content/drive/MyDrive/Models/vgg_model.keras')

test_datagen = ImageDataGenerator(rescale=1.0 / 255)
test_generator = test_datagen.flow_from_directory(
    '/content/Dataset/test',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False,
)

true_labels = test_generator.classes

test_steps_per_epoch = np.math.ceil(
    test_generator.samples / test_generator.batch_size
)

predictions1 = model1.predict(
    test_generator,
    steps=test_steps_per_epoch,
)
predictions1_binary = np.argmax(predictions1, axis=1)

predictions2 = model2.predict(
    test_generator,
    steps=test_steps_per_epoch,
)
predictions2_binary = np.argmax(predictions2, axis=1)

f1_score1 = f1_score(
    true_labels,
    predictions1_binary,
    average='weighted',
)
f1_score2 = f1_score(
    true_labels,
    predictions2_binary,
    average='weighted',
)

data = {
    'Model': ['Model 1', 'Model 2'],
    'F1 Score': [f1_score1, f1_score2],
}

comparison_df = pd.DataFrame(data)

excel_path = '/content/drive/MyDrive/model_comparison(2).xlsx'
comparison_df.to_excel(excel_path, index=False)

print("F1 scores comparison saved to Excel.")

## Supplementary evaluation - not present in Appendix C

The paper discusses model accuracy and confusion matrices, but the printed Appendix C does not include the corresponding calculations. The following cell completes those evaluations using the predictions already generated above.

In [ ]:
from sklearn.metrics import accuracy_score

# Accuracy
resnet_accuracy = accuracy_score(true_labels, predictions1_binary)
vgg_accuracy = accuracy_score(true_labels, predictions2_binary)

print(f"ResNet50 accuracy: {resnet_accuracy:.4f}")
print(f"VGG16 accuracy:    {vgg_accuracy:.4f}")
print(f"ResNet50 weighted F1: {f1_score1:.4f}")
print(f"VGG16 weighted F1:    {f1_score2:.4f}")

# Confusion matrices
resnet_confusion = confusion_matrix(true_labels, predictions1_binary)
vgg_confusion = confusion_matrix(true_labels, predictions2_binary)

class_names = [
    name
    for name, index in sorted(
        test_generator.class_indices.items(),
        key=lambda item: item[1],
    )
]

resnet_confusion_df = pd.DataFrame(
    resnet_confusion,
    index=class_names,
    columns=class_names,
)

vgg_confusion_df = pd.DataFrame(
    vgg_confusion,
    index=class_names,
    columns=class_names,
)

print("\nResNet50 confusion matrix:")
display(resnet_confusion_df)

print("VGG16 confusion matrix:")
display(vgg_confusion_df)

## Reported results in the paper

| Model | Training accuracy | Test accuracy | Weighted F1 |
|---|---:|---:|---:|
| ResNet50 | 82.0% | 74.83% | 0.7158 |
| VGG16 | 79.0% | 72.7% | 0.7119 |

These are the values reported in the accompanying paper, not outputs precomputed in this notebook.